# Task 4

## Task: Banknote Authentication Classification

In this task, you will work with the dataset provided in the file **`data_banknote_authentication.csv`**

Your objective is to build and evaluate classification models to predict the authenticity of banknotes.

You should:

- Load and explore the dataset.
- Build two classification models:
  - **Decision Tree**
  - **Random Forest**
- Use **GridSearchCV** to optimize the hyperparameters of each model.
- Evaluate the performance of both models using:
  - **Confusion Matrix**
  - **Classification Report**

Finally, **compare the performance of the two models** and discuss which model performs better for this dataset.

In [1]:
import pandas as pd

df = pd.read_csv("data_banknote_authentication.csv")


c:\Users\nayer\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\nayer\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df.head()

,Variance_Wavelet,Skewness_Wavelet,Curtosis_Wavelet,Image_Entropy,Class
0,3.62160,8.6661,-2.8073,-0.44699,0
1,4.54590,8.1674,-2.4586,-1.46210,0
2,3.86600,-2.6383,1.9242,0.10645,0
3,3.45660,9.5228,-4.0112,-3.59440,0
4,0.32924,-4.4552,4.5718,-0.98880,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1372 entries, 0 to 1371
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Variance_Wavelet  1372 non-null   float64
 1   Skewness_Wavelet  1372 non-null   float64
 2   Curtosis_Wavelet  1372 non-null   float64
 3   Image_Entropy     1372 non-null   float64
 4   Class             1372 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 53.7 KB


In [5]:
df.describe()

,Variance_Wavelet,Skewness_Wavelet,Curtosis_Wavelet,Image_Entropy,Class
count,1372.000000,1372.000000,1372.000000,1372.000000,1372.000000
mean,0.433735,1.922353,1.397627,-1.191657,0.444606
std,2.842763,5.869047,4.310030,2.101013,0.497103
min,-7.042100,-13.773100,-5.286100,-8.548200,0.000000
25%,-1.773000,-1.708200,-1.574975,-2.413450,0.000000
50%,0.496180,2.319650,0.616630,-0.586650,0.000000
75%,2.821475,6.814625,3.179250,0.394810,1.000000
max,6.824800,12.951600,17.927400,2.449500,1.000000


In [ ]:
#separate the features into input features X and target/output Y (class)
X = df.drop('Class', axis=1)
y = df['Class']

In [ ]:
#split the data into train/test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=101
)

In [9]:
#GridSearch to try different hyper-parameters and auto-selects the best combination
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

dt = DecisionTreeClassifier()
#maximum depth of tree, min samples needed to split a node, method used to measure split quality(gini, entropy)
param_grid_dt = {
    'max_depth': [None, 2, 4, 6, 8],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

In [10]:
grid_dt = GridSearchCV(dt, param_grid_dt, cv=5) #GridSearch Method
#5-fold cross-validation
grid_dt.fit(X_train, y_train)

best_dt = grid_dt.best_estimator_ #gives the best trained model
print("Best Decision Tree Parameters:", grid_dt.best_params_) #shows best parameter combinations

Best Decision Tree Parameters: {'criterion': 'entropy', 'max_depth': 6, 'min_samples_split': 5}


In [ ]:
#predict the test data and compare predictions with actual values
from sklearn.metrics import classification_report, confusion_matrix

dt_preds = best_dt.predict(X_test)

print("Decision Tree Results:\n")
print(confusion_matrix(y_test, dt_preds))
print("\n")
print(classification_report(y_test, dt_preds))

Decision Tree Results:

[[237   1]
 [  1 173]]


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       238
           1       0.99      0.99      0.99       174

    accuracy                           1.00       412
   macro avg       1.00      1.00      1.00       412
weighted avg       1.00      1.00      1.00       412



Random Forest Model:


In [12]:
#build and tune
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
#number of trees in a forest, max depth of each, number of features considered at each split
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 4, 6, 8],
    'max_features': ['sqrt', 'log2']
}

In [13]:
grid_rf = GridSearchCV(rf, param_grid_rf, cv=5)
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_
print("Best Random Forest Parameters:", grid_rf.best_params_)

Best Random Forest Parameters: {'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 100}


In [14]:
#test the best Random Forest model on the test data.
rf_preds = best_rf.predict(X_test)

print("Random Forest Results:\n")
print(confusion_matrix(y_test, rf_preds))
print("\n")
print(classification_report(y_test, rf_preds))

Random Forest Results:

[[234   4]
 [  0 174]]


              precision    recall  f1-score   support

           0       1.00      0.98      0.99       238
           1       0.98      1.00      0.99       174

    accuracy                           0.99       412
   macro avg       0.99      0.99      0.99       412
weighted avg       0.99      0.99      0.99       412



### Model Comparison

- The Decision Tree achieved slightly higher accuracy (almost 100%) with only 2 misclassifications.
- The Random Forest achieved slightly lower accuracy (99%) with 4 misclassifications.
- However, Decision Trees are more prone to overfitting, while Random Forest provides better generalization by combining multiple trees.

### Conclusion

Although the Decision Tree performed slightly better on the test set, the Random Forest model is preferred because it is more robust and less likely to overfit.